In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CATALOG         = "intelligent_etl"
SCHEMA          = "default"
ALERT_THRESHOLD = 0.10  # alert if anomaly rate > 10%

clean      = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_clean")
quarantine = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_quarantine")

total        = clean.count() + quarantine.count()
anomalies    = quarantine.count()
anomaly_rate = anomalies / total

print("="*50)
print("PIPELINE HEALTH CHECK")
print("="*50)
print(f"  Timestamp:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Total records:  {total:,}")
print(f"  Clean:          {clean.count():,}")
print(f"  Anomalies:      {anomalies:,}")
print(f"  Anomaly rate:   {anomaly_rate:.1%}")
print(f"  Threshold:      {ALERT_THRESHOLD:.1%}")
print(f"  Status:         {'⚠️  ALERT' if anomaly_rate > ALERT_THRESHOLD else '✅  OK'}")
print("="*50)

PIPELINE HEALTH CHECK
  Timestamp:      2026-05-13 13:35:20
  Total records:  9,879
  Clean:          9,161
  Anomalies:      718
  Anomaly rate:   7.3%
  Threshold:      10.0%
  Status:         ✅  OK


In [0]:
from pyspark.sql import functions as F

q = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_quarantine")

print("Anomaly breakdown by signal:")
print("-"*40)

rule_count = q.filter(F.col("rule_flag")   == True).count()
zscore_count = q.filter(F.col("zscore_flag") == True).count()
if_count   = q.filter(F.col("if_flag")     == True).count()

print(f"  Rule-based:       {rule_count:>5,} rows")
print(f"  Z-score:          {zscore_count:>5,} rows")
print(f"  Isolation Forest: {if_count:>5,} rows")
print(f"  Total unique:     {anomalies:>5,} rows")

print("\nTop anomaly reasons:")
(q
 .groupBy("anomaly_reason")
 .count()
 .orderBy(F.col("count").desc())
 .show(10, truncate=False))

Anomaly breakdown by signal:
----------------------------------------
  Rule-based:         320 rows
  Z-score:            143 rows
  Isolation Forest:   494 rows
  Total unique:       718 rows

Top anomaly reasons:
+------------------------------------------+-----+
|anomaly_reason                            |count|
+------------------------------------------+-----+
|RULE:dq_future_date                       |115  |
|RULE:dq_invalid_amount                    |99   |
|RULE:dq_has_required_null,ZSCORE:max=8.693|9    |
|IF:score=-0.6293                          |5    |
|IF:score=-0.6275                          |4    |
|IF:score=-0.6327                          |3    |
|IF:score=-0.6439                          |3    |
|IF:score=-0.6273                          |3    |
|IF:score=-0.6398                          |3    |
|IF:score=-0.6474                          |3    |
+------------------------------------------+-----+
only showing top 10 rows


In [0]:
# How much revenue is sitting in quarantine?
q = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_quarantine")
c = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_clean")

quarantine_revenue = q.agg(
    F.sum("total_amount").alias("quarantine_revenue"),
    F.avg("total_amount").alias("avg_quarantine_order")
).collect()[0]

clean_revenue = c.agg(
    F.sum("total_amount").alias("clean_revenue"),
    F.avg("total_amount").alias("avg_clean_order")
).collect()[0]

print("Revenue Impact Analysis:")
print("="*45)
print(f"  Clean revenue:          ₹{clean_revenue['clean_revenue']:>15,.2f}")
print(f"  Quarantine revenue:     ₹{quarantine_revenue['quarantine_revenue']:>15,.2f}")
print(f"  Avg clean order:        ₹{clean_revenue['avg_clean_order']:>15,.2f}")
print(f"  Avg quarantine order:   ₹{quarantine_revenue['avg_quarantine_order']:>15,.2f}")
print(f"\n  Quarantine avg is higher because price outliers")
print(f"  (10-50x normal price) pull the average up.")

Revenue Impact Analysis:
  Clean revenue:          ₹ 225,787,209.55
  Quarantine revenue:     ₹ 204,788,263.00
  Avg clean order:        ₹      24,646.57
  Avg quarantine order:   ₹     285,220.42

  Quarantine avg is higher because price outliers
  (10-50x normal price) pull the average up.


In [0]:
import mlflow

mlflow.set_experiment("/intelligent_etl_anomaly_detection")

with mlflow.start_run(run_name=f"pipeline_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    mlflow.log_metrics({
        "total_records":   total,
        "clean_records":   clean.count(),
        "anomaly_count":   anomalies,
        "anomaly_rate":    round(anomaly_rate, 4),
        "rule_flagged":    rule_count,
        "zscore_flagged":  zscore_count,
        "if_flagged":      if_count,
    })
    mlflow.log_params({
        "alert_threshold": ALERT_THRESHOLD,
        "catalog":         CATALOG,
        "schema":          SCHEMA,
    })
    print("Metrics logged to MLflow ✅")
    print(f"  anomaly_rate: {anomaly_rate:.4f}")
    print(f"  total_records: {total}")

Metrics logged to MLflow ✅
  anomaly_rate: 0.0727
  total_records: 9879
